# Integrating evolutionary and epidemiological signals to uncover superspreading dynamics


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import string

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

from matplotlib.axes import Axes
from matplotlib.figure import Figure
from plottable import Table
from scipy import stats

FIGS_DIR = Path("../figures")
TABLES_DIR = Path("../tables")
SCOVMOD_DIR = Path("../data/processed/synthetic/scovmod")
FORMATS = ("pdf",)

MODELS = {
    "LinearDistScore": "DDS",
    "PoissonDistScore": "SDS",
    "ProbLinearDist": "DDP",
    "ProbPoissonDist": "SDP",
    "LogitLinearDist10": "DDL1",
    "LogitPoissonDist10": "SDL1",
    "LogitLinearDist100": "DDL2",
    "LogitPoissonDist100": "SDL2",
}

SCENARIOS = {
    "baseline": "Baseline",
    "surveillance_moderate": "Surveillance (moderate)",
    "surveillance_severe": "Surveillance (severe)",
    "low_clock_signal": "Low clock signal",
    "low_incubation_shape": "Low incubation shape",
    "low_incubation_scale": "Low incubation scale",
    "high_clock_signal": "High clock signal",
    "high_incubation_shape": "High incubation shape",
    "high_incubation_scale": "High incubation scale",
    "relaxed_clock": "Relaxed clock",
    "adversarial": "Adversarial",
}

MODEL_ORDER = list(MODELS.values())
SCENARIO_ORDER = list(SCENARIOS.values())

# Colorblind-safe palette (Okabe-Ito inspired)
COLORS = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "green": "#009E73",
    "vermillion": "#D55E00",
    "sky": "#56B4E9",
    "purple": "#CC79A7",
    "yellow": "#F0E442",
    "gray": "#7F7F7F",
    "black": "#1F1F1F",
}

STAGE_COLORS = {
    "Latent": COLORS["green"],
    "Presymptomatic": COLORS["orange"],
    "Symptomatic": COLORS["purple"],
}

GENETIC_COLORS = {
    "Raw": COLORS["gray"],
    "Normalised": COLORS["green"],
}

TREE_STEP_COLORS = {
    "raw": COLORS["gray"],
    "cleaned": COLORS["sky"],
    "selected_component": COLORS["orange"],
    "final_tree": COLORS["vermillion"],
}

MODEL_COLORS = {
    "DDP": COLORS["blue"],
    "DDL2": COLORS["orange"],
    "SDP": COLORS["green"],
    "SDL2": COLORS["purple"],
}

STABILITY_COLORS = {
    "forward": COLORS["blue"],
    "backward": COLORS["orange"],
    "jaccard": COLORS["green"],
}


def set_plot_theme(font_scale: float = 1.2) -> None:
    sns.set_theme(
        context="paper",
        style="whitegrid",
        font_scale=font_scale,
        rc={
            "figure.dpi": 500,
            "savefig.dpi": 500,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.edgecolor": "#333333",
            "grid.alpha": 0.25,
            "grid.linewidth": 0.6,
            "legend.frameon": False,
        },
    )


set_plot_theme()


In [ ]:
def save_figure(fig: Figure, out_base: Path, formats: tuple[str, ...] = FORMATS) -> None:
    out_base.parent.mkdir(parents=True, exist_ok=True)
    for ext in formats:
        fig.savefig(out_base.with_suffix(f".{ext}"), bbox_inches="tight", dpi=500)


def read_table(stem: str) -> pd.DataFrame:
    return pd.read_parquet(TABLES_DIR / f"{stem}.parquet")


def read_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def add_panel_labels(
    axes: list[Axes],
    labels: list[str],
    x: float = -0.16,
    y: float = 1.08,
    size: float = 12,
) -> None:
    for ax, label in zip(axes, labels):
        ax.text(
            x,
            y,
            f"{label})",
            transform=ax.transAxes,
            fontsize=size,
            fontweight="bold",
            va="top",
        )


def style_colorbar(heatmap_artist, label_size: int = 10) -> None:
    cbar = heatmap_artist.collections[0].colorbar
    cbar.outline.set_edgecolor("#333333")
    cbar.outline.set_linewidth(0.8)
    cbar.ax.yaxis.label.set_size(label_size)


def ccdf(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.array([]), np.array([])
    arr = np.sort(arr.astype(int))
    unique, first_idx = np.unique(arr, return_index=True)
    return unique, (arr.size - first_idx) / arr.size


def tree_depths(graph: nx.DiGraph) -> np.ndarray:
    roots = [node for node, indegree in graph.in_degree(graph.nodes) if indegree == 0]
    if not roots:
        raise ValueError("Tree has no roots (in-degree 0).")

    depth_map: dict[object, int] = {}
    for root in roots:
        for node, dist in nx.single_source_shortest_path_length(graph, root).items():
            if node not in depth_map or dist < depth_map[node]:
                depth_map[node] = dist
    return np.asarray(list(depth_map.values()), dtype=int)


def metric_label(metric: str) -> str:
    labels = {
        "BCubed_F1_Score": "BCubed F1 score",
        "BCubed_Precision": "BCubed precision",
        "BCubed_Recall": "BCubed recall",
    }
    return labels.get(metric, metric.replace("_", " "))


## 1) Features of *`epilink`*


In [ ]:
# Load data
toit_df = read_table("characterise_epilink/characteristic_toit_grid")
toit_df = toit_df[toit_df["days"] <= 15]

tost_df = read_table("characterise_epilink/characteristic_tost_grid")
tost_df = tost_df[tost_df["days"].between(-10, 10)]

presymptomatic_fraction_df = read_table("characterise_epilink/characteristic_presymptomatic_fraction")
stage_df = read_table("characterise_epilink/characteristic_stage_samples")
clock_rates_df = read_table("characterise_epilink/characteristic_clock_rate_samples")
temporal_df = read_table("characterise_epilink/characteristic_temporal_linkage")
genetic_df = read_table("characterise_epilink/characteristic_genetic_linkage")
surface_df = read_table("characterise_epilink/characteristic_probability_surface")
scenarios_df = read_table("characterise_epilink/characteristic_genetic_scenarios")

surface_pivot = surface_df.pivot(index="days", columns="snp", values="probability")
scenario_pivot = scenarios_df.pivot(index="m", columns="snp", values="normalized")

# Derived quantities
presymptomatic_fraction = float(presymptomatic_fraction_df.iloc[0, 0])
latent = stage_df.loc[stage_df["stage"] == "latent", "value"].to_numpy()
presym = stage_df.loc[stage_df["stage"] == "presymptomatic", "value"].to_numpy()
sympt = stage_df.loc[stage_df["stage"] == "symptomatic", "value"].to_numpy()

disease_dynamic = pd.DataFrame(
    {
        "Latent": latent,
        "Presymptomatic": latent + presym,
        "Symptomatic": latent + presym + sympt,
    }
).melt(var_name="Stage", value_name="Duration")
disease_dynamic = disease_dynamic[disease_dynamic["Duration"] <= 15]


In [ ]:
fig = plt.figure(figsize=(12, 10), constrained_layout=True)
grid = fig.add_gridspec(3, 3)

ax1 = fig.add_subplot(grid[0, 0])
ax2 = fig.add_subplot(grid[0, 1])
ax3 = fig.add_subplot(grid[0, 2])
ax4 = fig.add_subplot(grid[1, 0])
ax5 = fig.add_subplot(grid[1, 1])
ax6 = fig.add_subplot(grid[1, 2])
ax7 = fig.add_subplot(grid[2, 0])
ax8 = fig.add_subplot(grid[2, 1])
ax9 = fig.add_subplot(grid[2, 2])

# A) TOIT
sns.lineplot(data=toit_df, x="days", y="pdf", ax=ax1, color=COLORS["blue"], linewidth=2.2)
ax1.axvline(0, color="#333333", linestyle=":", linewidth=1.2)
ax1.set(xlabel="TOIT (days)", ylabel="Density")

# B) TOST
sns.lineplot(data=tost_df, x="days", y="pdf", ax=ax2, color=COLORS["vermillion"], linewidth=2.2)
ax2.axvline(0, color="#333333", linestyle=":", linewidth=1.2)
ax2.set(xlabel="TOST (days)", ylabel="Density")

# C) Disease stage progression
stage_styles = {"Latent": "-", "Presymptomatic": "--", "Symptomatic": "-."}
for stage, linestyle in stage_styles.items():
    subset = disease_dynamic[disease_dynamic["Stage"] == stage]
    if subset.empty:
        continue
    sns.kdeplot(
        data=subset,
        x="Duration",
        ax=ax3,
        color=STAGE_COLORS[stage],
        linewidth=2,
        linestyle=linestyle,
        fill=False,
        label=stage,
    )
ax3.set(xlabel="Infection progression (days)", ylabel="Density")
ax3.legend(loc="best", title=None, handlelength=3)

# D) Substitution rate distribution
sns.kdeplot(
    data=clock_rates_df,
    x="rate_per_site_year",
    ax=ax4,
    color=COLORS["green"],
    linewidth=2.2,
    fill=False,
)
ax4.set(xlabel="Substitution rate (site/year)", ylabel="Density")

# E) Presymptomatic card
ax5.axis("off")
ax5.text(
    0.5,
    0.62,
    f"{presymptomatic_fraction:.1%}",
    ha="center",
    va="center",
    fontsize=30,
    fontweight="bold",
    color=COLORS["blue"],
    bbox={"boxstyle": "round,pad=0.5", "fc": "#EAF2F8", "ec": COLORS["blue"], "lw": 1.2},
)
ax5.text(0.5, 0.28, "Presymptomatic fraction", ha="center", va="center", fontsize=13)

# F) Temporal component
sns.lineplot(
    data=temporal_df,
    x="days",
    y="probability",
    ax=ax6,
    color=COLORS["blue"],
    linewidth=2.2,
)
ax6.set(xlabel="Temporal distance (days)", ylabel="Probability")

# G) Genetic component
sns.lineplot(
    data=genetic_df,
    x="snp",
    y="normalized",
    ax=ax7,
    color=GENETIC_COLORS["Normalised"],
    linewidth=2,
)
ax7.set(xlabel="Genetic distance (SNPs)", ylabel="Probability")

# H) Joint probability surface
hm8 = sns.heatmap(
    surface_pivot,
    cmap="mako",
    ax=ax8,
    cbar_kws={"label": "Probability"},
    linewidths=0,
)
ax8.set(xlabel="Genetic distance (SNPs)", ylabel="Temporal distance (days)")
style_colorbar(hm8)

# I) Scenario heatmap
hm9 = sns.heatmap(
    scenario_pivot,
    cmap="cividis",
    ax=ax9,
    cbar_kws={"label": "Probability"},
    linewidths=0,
)
ax9.set(xlabel="Genetic distance (SNPs)", ylabel="Intermediate hosts")
style_colorbar(hm9)

add_panel_labels(
    [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8, ax9],
    ["A", "B", "C", "D", "E", "F", "G", "H", "I"],
    x=-0.18,
    y=1.13,
    size=13
)

save_figure(fig, FIGS_DIR / "epilink_feature_summary")
plt.show()
plt.close(fig)

## 2) Characteristics of the transmission tree


In [ ]:
tree = nx.read_gml(SCOVMOD_DIR / "scovmod_tree.gml")
heterogeneity = read_json(SCOVMOD_DIR / "scovmod_tree_tree_heterogeneity.json")

tree_out_deg = np.array([deg for _, deg in tree.out_degree(tree.nodes)], dtype=int)
tree_depth = tree_depths(tree)

tree_summary_df = read_table("scovmod/scovmod_tree_summary")
tree_degree_df = read_table("scovmod/scovmod_tree_degree_distributions")
tree_component_df = read_table("scovmod/scovmod_tree_component_sizes")

tree_summary_final = tree_summary_df.loc[tree_summary_df["label"] == "final_tree"]
if tree_summary_final.empty:
    tree_summary_final = tree_summary_df.iloc[[-1]]
tree_summary_final = tree_summary_final.iloc[0]


In [ ]:
fig = plt.figure(figsize=(10, 9), constrained_layout=True)
grid = fig.add_gridspec(2, 2)

ax1 = fig.add_subplot(grid[0, 0])
ax2 = fig.add_subplot(grid[0, 1])
ax3 = fig.add_subplot(grid[1, 0])
ax4 = fig.add_subplot(grid[1, 1])

line_style_map = {
    "raw": ("--", "o"),
    "cleaned": ("-.", "s"),
    "selected_component": (":", "^"),
    "final_tree": ("-", "D"),
}

# A) Component-size CCDF
component_sizes = tree_component_df["component_size"].to_numpy(dtype=int)
x_comp, y_comp = ccdf(component_sizes)
ax1.plot(x_comp, y_comp, color=COLORS["blue"], linewidth=2.2)
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set(xlabel="Component size (nodes)", ylabel="CCDF")

# B) Out-degree CCDF across filtering steps
for label in ["raw", "cleaned", "selected_component"]:
    subset = tree_degree_df.loc[
        (tree_degree_df["graph"] == label) & (tree_degree_df["degree_type"] == "out"),
        "value",
    ]
    values = subset.to_numpy(dtype=int)
    values = values[values > 0]
    if values.size == 0:
        continue

    x_vals, y_vals = ccdf(values)
    linestyle, marker = line_style_map[label]
    ax2.plot(
        x_vals,
        y_vals,
        color=TREE_STEP_COLORS[label],
        linewidth=1.8,
        linestyle=linestyle,
        marker=marker,
        markersize=4,
        markevery=max(1, len(x_vals) // 12),
        label=label,
    )

final_nonzero = tree_out_deg[tree_out_deg > 0]
if final_nonzero.size > 0:
    x_tree, y_tree = ccdf(final_nonzero)
    linestyle, marker = line_style_map["final_tree"]
    ax2.plot(
        x_tree,
        y_tree,
        color=TREE_STEP_COLORS["final_tree"],
        linewidth=2.2,
        linestyle=linestyle,
        marker=marker,
        markersize=4,
        markevery=max(1, len(x_tree) // 12),
        label="final_tree",
    )

ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set(xlabel="Out-degree (offspring)", ylabel="CCDF")
ax2.legend(loc="best", title=None, handlelength=3)

# C) Tree depth distribution
counts = pd.Series(tree_depth).value_counts().sort_index()
bars = ax3.bar(
    counts.index,
    counts.values,
    color="#DDEAF7",
    edgecolor=COLORS["blue"],
    linewidth=0.9,
    width=0.9,
)
for patch in bars:
    patch.set_hatch("//")
ax3.set_yscale("log")
ax3.set(xlabel="Depth from root", ylabel="Nodes")

summary_text = (
    f"n={int(tree_summary_final['n_nodes']):,}\n"
    f"edges={int(tree_summary_final['n_edges']):,}\n"
    f"max out-degree={int(tree_summary_final['max_out_degree'])}\n"
    f"mean out-degree={tree_summary_final['mean_out_degree']:.2f}"
)
ax3.text(
    0.98,
    0.98,
    summary_text,
    transform=ax3.transAxes,
    ha="right",
    va="top",
    fontsize=10,
    bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "#444444", "alpha": 0.95},
)

# D) Offspring distribution and NB fit
x_max = int(np.quantile(tree_out_deg, 0.99))
x_max = max(20, min(x_max, int(tree_out_deg.max())))
bins = np.arange(-0.5, x_max + 1.5, 1.0)

sns.histplot(
    tree_out_deg,
    bins=bins,
    stat="probability",
    color="#F8E4CC",
    edgecolor=COLORS["orange"],
    linewidth=0.9,
    ax=ax4,
)

mean_rt = float(heterogeneity.get("meanRt", np.nan))
disp_k = float(heterogeneity.get("disp_k", np.nan))
if np.isfinite(mean_rt) and np.isfinite(disp_k) and disp_k > 0:
    x_vals = np.arange(0, x_max + 1, dtype=int)
    p = disp_k / (disp_k + mean_rt)
    pmf = stats.nbinom.pmf(x_vals, disp_k, p)
    ax4.plot(x_vals, np.clip(pmf, 1e-12, None), color=COLORS["vermillion"], linewidth=2.2, label="NB fit")
    ax4.legend(loc="best")

ax4.set_yscale("log")
ax4.set(xlabel="Offspring count (out-degree)", ylabel="Probability")

het_text = (
    f"R={mean_rt:.2f}\n"
    f"k={disp_k:.2f}\n"
    f"Zero={heterogeneity.get('pct_zero_transmitters', np.nan):.1f}%\n"
    f"80% by {heterogeneity.get('prop_80_percent_transmitters', np.nan) * 100:.1f}%"
)
ax4.text(
    0.98,
    0.98,
    het_text,
    transform=ax4.transAxes,
    ha="right",
    va="top",
    fontsize=10,
    bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "#444444", "alpha": 0.95},
)

add_panel_labels(
    [ax1, ax2, ax3, ax4],
    ["A", "B", "C", "D"],
    x=-0.12,
    y=1.08,
)

save_figure(fig, FIGS_DIR / "sm_scovmod_tree_overview")
plt.show()
plt.close(fig)

## 3) Pairwise discrimination and cluster recovery


In [ ]:
discrimination = read_table("discrimination/discrimination_metrics")
clustering_metrics = read_table("clustering/clustering_metrics")
clustering_stability = read_table("clustering/clustering_stability")

discrimination["ModelLabel"] = discrimination["Model"].map(MODELS)
discrimination["ScenarioLabel"] = discrimination["Scenario"].map(SCENARIOS)

clustering_metrics["ModelLabel"] = clustering_metrics["Weight_Column"].map(MODELS)
clustering_metrics["ScenarioLabel"] = clustering_metrics["Scenario"].map(SCENARIOS)

clustering_stability["ModelLabel"] = clustering_stability["Weight_Column"].map(MODELS)
clustering_stability["ScenarioLabel"] = clustering_stability["Scenario"].map(SCENARIOS)

pr_heat = discrimination.pivot(
    index="ScenarioLabel",
    columns="ModelLabel",
    values="PR_AUC",
).reindex(index=SCENARIO_ORDER, columns=MODEL_ORDER)

best_f1 = (
    clustering_metrics.groupby(["ScenarioLabel", "ModelLabel"], as_index=False)["BCubed_F1_Score"]
    .max()
)
best_f1_heat = best_f1.pivot(
    index="ScenarioLabel",
    columns="ModelLabel",
    values="BCubed_F1_Score",
).reindex(index=SCENARIO_ORDER, columns=MODEL_ORDER)
best_f1_heat.dropna(axis=1, how='all', inplace=True)

In [ ]:
fig = plt.figure(figsize=(10, 6))
grid = fig.add_gridspec(1, 2, width_ratios=[2, 1])

ax1 = fig.add_subplot(grid[0])
ax2 = fig.add_subplot(grid[1])

hm1 = sns.heatmap(
    pr_heat,
    vmin=0,
    vmax=1,
    cmap="YlGnBu",
    linewidths=0.1,
    annot=True,
    fmt=".2f",
    cbar_kws={"label": "Average Precision (AP)"},
    ax=ax1,
)
ax1.set(xlabel="", ylabel="")
style_colorbar(hm1)

hm2 = sns.heatmap(
    best_f1_heat,
    vmin=0,
    vmax=1,
    cmap="YlGnBu",
    linewidths=0.1,
    annot=True,
    fmt=".2f",
    cbar_kws={"label": r"Best F1 score"},
    ax=ax2,
)
ax2.set(xlabel="", ylabel="")
ax2.set_yticklabels([])
style_colorbar(hm2)

add_panel_labels([ax1, ax2], ["A", "B"])

plt.tight_layout()
save_figure(fig, FIGS_DIR / "discrimination_recovery")
plt.show()
plt.close(fig)

### Supplementary: metric profiles across resolution


In [ ]:
def plot_cluster_metrics(
    df: pd.DataFrame,
    out_path: Path,
    metric: str = "BCubed_F1_Score",
    y_axis: str = "Resolution",
    ncols: int = 3,
    figsize: tuple[float, float] = (12, 12),
) -> None:
    model_subset = ["DDP", "DDL2", "SDP", "SDL2"]
    line_styles = {"DDP": "-", "DDL2": "--", "SDP": ":", "SDL2": "-."}
    markers = {"DDP": "o", "DDL2": "s", "SDP": "^", "SDL2": "D"}

    scenarios = [scenario for scenario in SCENARIO_ORDER if scenario in set(df["ScenarioLabel"])]
    panel_count = len(scenarios)
    nrows = math.ceil(panel_count / ncols)

    fig = plt.figure(figsize=figsize, constrained_layout=True)
    grid = fig.add_gridspec(nrows, ncols)

    axes: list[Axes] = []
    for idx in range(nrows * ncols):
        r, c = divmod(idx, ncols)
        ax = fig.add_subplot(grid[r, c], sharey=axes[0] if axes else None)
        axes.append(ax)

    legend_handles = None
    legend_labels = None

    for idx, scenario in enumerate(scenarios):
        ax = axes[idx]
        subset = df[df["ScenarioLabel"] == scenario]

        for model in model_subset:
            rows = subset[subset["ModelLabel"] == model]
            if rows.empty:
                continue
            ax.plot(
                rows[y_axis],
                rows[metric],
                color=MODEL_COLORS[model],
                linestyle=line_styles[model],
                marker=markers[model],
                linewidth=1.7,
                markersize=4.5,
                label=model,
            )

        if legend_handles is None:
            legend_handles, legend_labels = ax.get_legend_handles_labels()

        ax.set_title(scenario, fontsize=11, pad=6, loc="left")
        ax.set_ylim((0, 1.05))

        if (idx % ncols) != 0:
            ax.set_ylabel("")
        if idx < (nrows - 1) * ncols:
            ax.set_xlabel("")

        panel_letter = string.ascii_uppercase[idx]
        ax.text(-0.18, 1.12, f"{panel_letter})", transform=ax.transAxes, fontsize=13, fontweight="bold", va="top")

    for idx in range(panel_count, nrows * ncols):
        axes[idx].axis("off")

    fig.supxlabel("Resolution", fontsize=13)
    fig.supylabel(metric_label(metric), fontsize=13)

    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            title="Model",
            fontsize=12,
            title_fontsize=13,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.08),
            ncol=4,
            frameon=False,
            handlelength=3,
        )

    save_figure(fig, out_path)
    plt.show()
    plt.close(fig)

In [ ]:
plot_cluster_metrics(
    clustering_metrics,
    out_path=FIGS_DIR / "sm_cluster_metrics_f1",
    metric="BCubed_F1_Score",
)

plot_cluster_metrics(
    clustering_metrics,
    out_path=FIGS_DIR / "sm_cluster_metrics_precision",
    metric="BCubed_Precision",
)

plot_cluster_metrics(
    clustering_metrics,
    out_path=FIGS_DIR / "sm_cluster_metrics_recall",
    metric="BCubed_Recall",
)

plot_cluster_metrics(
    clustering_stability,
    out_path=FIGS_DIR / "sm_clustering_stability_f1",
    metric="BCubed_F1_Score",
    y_axis="Res1",
)


## 4) Temporal stability of clusters


In [ ]:
def plot_stability_over_time(stability: pd.DataFrame, ax: Axes) -> Axes:
    line_styles = {"forward": "--", "backward": ":", "jaccard": "-"}
    markers = {"forward": "o", "backward": "s", "jaccard": "^"}

    for metric in ["forward", "backward", "jaccard"]:
        ax.plot(
            stability.index,
            stability[metric],
            linestyle=line_styles[metric],
            linewidth=1.8,
            color=STABILITY_COLORS[metric],
            markersize=4,
            marker=markers[metric],
            label=metric.capitalize(),
        )

    ax.set(xlabel="Epidemic week", ylabel="Mean stability")
    ax.set_ylim(0, 1)
    ax.legend(loc="lower right", title=None, handlelength=3)
    return ax


def plot_cumulative_cases(cases: pd.DataFrame, ax: Axes) -> Axes:
    cases = cases.copy()
    cases["cumulative"] = cases["n_cases"].cumsum()

    b = ax.bar(
        cases["available_time"],
        cases["cumulative"],
        color="#DDEAF7",
        edgecolor=COLORS["blue"],
        linewidth=0.9,
        width=0.9,
    )
    for pt in b:
        pt.set_hatch("..")

    ax.set(xlabel="Epidemic week", ylabel="Cumulative cases")
    ax.set_yscale("log")
    return ax


data_accrual = read_table("temporal_stability/case_counts_over_time")
temporal_stability_ll = read_table("temporal_stability/temporal_stability_logit_linear")
temporal_stability_lp = read_table("temporal_stability/temporal_stability_logit_poisson")
temporal_stability_pl = read_table("temporal_stability/temporal_stability_prob_linear")
temporal_stability_pp = read_table("temporal_stability/temporal_stability_prob_poisson")


In [ ]:
fig = plt.figure(figsize=(10, 5))
grid = fig.add_gridspec(1, 2)

ax1 = fig.add_subplot(grid[0])
ax2 = fig.add_subplot(grid[1])

ax1 = plot_cumulative_cases(data_accrual, ax1)
ax2 = plot_stability_over_time(temporal_stability_pp, ax2)

add_panel_labels([ax1, ax2], ["A", "B"],
                 x=-0.14, y=1.08, size=11)

plt.tight_layout()
save_figure(fig, FIGS_DIR / "temporal_stability")
plt.show()
plt.close(fig)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=True)

ax1 = plot_stability_over_time(temporal_stability_pl, axes[0, 0])
ax2 = plot_stability_over_time(temporal_stability_pp, axes[0, 1])
ax3 = plot_stability_over_time(temporal_stability_ll, axes[1, 0])
ax4 = plot_stability_over_time(temporal_stability_lp, axes[1, 1])

for axis in [ax1, ax2]:
    axis.set_xlabel("")

add_panel_labels([ax1, ax2, ax3, ax4], ["A", "B", "C", "D"], x=-0.14, y=1.08, size=11)

plt.tight_layout()
save_figure(fig, FIGS_DIR / "sm_temporal_stability")
plt.show()
plt.close(fig)

## 5) Recovery of Boston SSE clusters


In [ ]:
boston_clusters_comp = read_table("boston/boston_cluster_composition")
boston_clusters_sum = read_table("boston/boston_cluster_summary")

exposure_cols = [
    "count::BHCHP",
    "count::Other",
    "count::City",
    "count::Conference",
    "count::SNF",
]

exposure_counts = boston_clusters_comp[exposure_cols].copy()
exposure_counts = exposure_counts.rename(
    columns={
        "count::BHCHP": "BHCHP",
        "count::Other": "Other",
        "count::City": "City",
        "count::Conference": "Conference",
        "count::SNF": "SNF",
    }
)
exposure_counts.index = boston_clusters_comp["cluster_id"]

boston_clusters_sum = boston_clusters_sum.drop(columns=["Size"]).copy()
boston_clusters_sum = boston_clusters_sum.set_index("Cluster ID").round(2).T
boston_clusters_sum.index.name = "Cluster ID"


In [ ]:
fig = plt.figure(figsize=(10, 5))
grid = fig.add_gridspec(1, 2, width_ratios=[1.5, 1])

ax1 = fig.add_subplot(grid[0])
ax2 = fig.add_subplot(grid[1])

hm = sns.heatmap(
    exposure_counts.T,
    vmin=0,
    vmax=80,
    cmap="YlOrBr",
    annot=True,
    fmt=".0f",
    annot_kws={"size": 10},
    cbar_kws={"label": "Cases", "shrink": 0.9},
    ax=ax1,
)
ax1.tick_params(axis="y", labelsize=12, rotation=0)
ax1.tick_params(axis="x", labelsize=11)
ax1.set_xlabel("Cluster ID")
style_colorbar(hm)

Table(boston_clusters_sum, ax=ax2)

add_panel_labels([ax1, ax2], ["A", "B"])

plt.tight_layout()
save_figure(fig, FIGS_DIR / "boston_cluster")
plt.show()
plt.close(fig)